# M0 Prompting for Success

This notebook demonstrates how AI responses vary and depend on the prompt provided by the user. If prompts are made without care, AI will return results that are undesirable and often incorrect. It is important to consider AI as a tool for reaching specific goals, such as "I would like to run a BLAST search for the following AGI loci: [X] using a bash script within a Jupyter Notebook ran on an HPC cluster. Can you provide me this bash script?", rather than a search engine that will provide you with an answer in response to a broad question, such as "How do I process RNA-Seq Data". 

The goal of this course is to teach students how to utilize command-line tools to process NGS data sets and display the results in R. Without a fundamental understanding of the biological context of the data, the experiment design, and the steps of processing NGS data after sequencing, prompting for help from AI tools will often lead to unfavorable results. Below are two examples of AI outputs generated using two separate prompts on OpenAI's GPT5 model. Please note the vast difference in the response quality: 

# Prompt #1: 
*"I am teaching a course on RNA-seq and CHiP/Dap-seq for graduate students in plant sciences. The course is run on an OnDemand hosted Jupyter notebook and remote VNC connection to our HPC cluster. These students will have to navigate the terminal, use programs in Linux for NGS techniques, as well as write bash scripts, and use R and R packages to complete the tasks of the course: (pseudo alignment, genome alignment, genome browser tracks, peak calling, DE analysis, etc). I want to write a notebook for the introduction that details the ways students can prompt AI with questions in the course to produce outputs specific to their needs. Can you write me this Jupyter notebook's .inpyb format?"*

# GPT-5 Response #1: 

## Introduction: How to use AI responsibly in this RNA‑seq / ChIP‑seq / DAP‑seq course

**Purpose:** This notebook explains how students should consult AI during the course. It covers good prompting, when to trust AI, how to verify answers, reproducibility practices, and ready‑to‑use prompt templates (bash, Slurm, R/DESeq2, peak calling, troubleshooting). Use this as the canonical guide for interacting with ChatGPT/other LLMs while working on the HPC and course assignments.

---

## Learning objectives

* Learn effective prompts to get accurate code, command lines, and explanations for NGS tasks.
* Understand AI limitations, safety, and privacy concerns when using cloud AI to analyze biological data.
* Learn reproducible practices: capturing software versions, citing AI outputs, and validating commands before running them.
* Use templates for common workflows: QC, pseudoalignment (Salmon), genome alignment (STAR/BWA), peak calling (MACS2), differential expression (DESeq2), and Slurm job scripts.

---

## Quick rules

1. **Always include system/context info** in prompts: OS(Alma Linux), scheduler (Slurm), available modules, conda/mamba env names, and exact package versions if known.
2. **Ask for (1) explanation, (2) suggested commands, (3) a dry‑run/printonly version**, and (4) a brief checklist of expected outputs. Don’t run generated commands blindly.
3. **Pin versions**: request `--version` checks in the script or a conda/mamba YAML to ensure reproducibility.
4. **Test on a small subset** (one or two FASTQ reads) before scaling to the full dataset.
5. **Cite AI use** in reports: include the prompt you used, the date, and which model/provider (e.g., ChatGPT, 2025‑10‑15). Treat AI like a helper, not an authority.
6. **Keep secrets out** of prompts: never paste credentials, private keys, or PHI.

---

## How to craft prompts — principles

* **Be specific**: Give file names, filepaths, sample structure, HPC limits (cores, mem), scheduler directives, and desired outputs.
* **Show what you tried**: paste the minimal command and error message — the AI can debug faster.
* **Ask for alternatives**: ask for 2–3 ways to do the step (e.g., salmon vs kallisto for pseudoalignment), with pros/cons.
* **Request safety checks**: ask the AI to highlight dangerous commands (e.g., `rm -rf`) and suggest safer alternatives.
* **Prefer code blocks**: ask for runnable code blocks and a commented explanation for each line.

---

## Prompt templates (copy & paste and fill placeholders)

### 1) Bash + Slurm job for preprocessing with `fastp` (template)

```bash
# Prompt to AI:
# "Create a Slurm sbatch script that runs `fastp` on paired FASTQ files stored under $HOME/project/fastq.
#  - Use 8 CPU cores and 32GB RAM, time 12:00:00, job name fastp_trim. Module system: module load fastp/0.23.1 (if modules unavailable use conda env 'ngs' with fastp==0.23.1). Output trimmed FASTQ to $HOME/project/trimmed. Provide a dry-run (echo) version and explain each line. Include version checks and an example command to run the script."
```

**What you should expect from AI:** a complete `sbatch` script with `#SBATCH` lines, module/conda activation, `fastp` command using `-i -I -o -O`, a step that prints `fastp --version`, and a `--json` summary path.

---

### 2) Salmon pseudoalignment quant (quick template)

```bash
# Prompt:
# "Given trimmed paired FASTQs in $HOME/project/trimmed, create a bash loop or Slurm script to run salmon quant (index already at $HOME/index/salmon_index) for all samples with 8 threads. Output to $HOME/project/salmon_quant/<sample>. Include a sample sheet parser step (tab-separated file samples.tsv with columns sample,fastq1,fastq2). Add checks for existing outputs and print commands instead of running when a --dryrun flag is given."
```

---

### 3) STAR alignment (genome) template

```bash
# Prompt:
# "Write a Slurm script to run STAR alignment for paired-end reads, using 16 threads, 64 GB RAM, and reference at /ref/Arabidopsis/STARindex.
#  - Use outSAMtype BAM SortedByCoordinate, output to $HOME/project/star_bam.
#  - Include samtools index step and samtools flagstat summary.
#  - Include version checks for STAR and samtools. Provide explanation and a checklist of expected output files."
```

---

### 4) Peak calling (MACS2) template for ChIP/DAP

```bash
# Prompt:
# "Produce a MACS2 callpeak command for narrow peaks with a treatment BAM and an input/control BAM. Assume Arabidopsis genome size is 1.35e8. Provide an sbatch header for 4 CPUs and 16GB, and request outputs: peaks.narrowPeak, summits.bed, and a bedGraph track convertible to bigWig. Include commands to make RPKM-normalized bigWig using deeptools bamCoverage."
```

---

### 5) DESeq2 analysis (R) template

```r
# Prompt:
# "Write an R script (usable as an R notebook cell) to:
#  1. Read Salmon quant files using tximport.
#  2. Build DESeqDataSet from a coldata table (columns: sample, condition, batch).
#  3. Run DESeq, output results table with shrinkage (lfcShrink), MA plot, PCA of vst-transformed counts, and write CSVs. Use DESeq2 version 1.38.0 and tximport 1.26.0. Provide code to save sessionInfo() and package versions."
```

---

## Safety & reproducibility checklist (ask the AI to return this in every code block)

1. Print `uname -a` and `date`.
2. Print `conda list --explicit` or `mamba list` or `module avail` depending on environment.
3. Print versions: `fastp --version`, `salmon --version`, `STAR --version`, `macs2 --version`, and R `sessionInfo()`.
4. Provide a `--dryrun`/`echo` mode in bash scripts.
5. Keep an output log (both STDOUT and STDERR) with timestamps: use `exec > >(tee -a logfile) 2>&1` or redirect in Slurm with `#SBATCH --output` and `--error`.
6. Add checksum or sample counts summary step (e.g., `fastqc`, `multiqc`) to verify the expected number of reads.

---

## Example: Asking AI to debug an error — best practice

1. Show the exact command you ran (copy/paste) and the full error message.
2. Show the `--version` of the tool and the environment (conda list or `module list`).
3. Ask the AI to suggest a minimal reproducer and explain why the error occurred.
4. Ask for safe fixes and one or two alternative commands.

**Bad prompt:** "my star command failed, fix it"

**Good prompt:**

```
I ran:
STAR --runThreadN 16 --genomeDir /ref/Arabidopsis/STARindex --readFilesIn sample_R1.fastq.gz sample_R2.fastq.gz --outFileNamePrefix sample_ --outSAMtype BAM SortedByCoordinate

I got this error: "ERROR: can't open genome file...". My environment: Jupyter Notebook Session hosted by RSS OnDemand on Hellbender, module system unavailable; I use conda env 'Module_3' with STAR v2.7.10a. `ls -l /ref/Arabidopsis/STARindex` shows ...

Please: 1) explain likely causes, 2) give commands to check the index integrity, 3) propose a fixed command and a dry-run version, 4) provide a Slurm script (sbatch) of the fixed workflow."
```

---

## How to ask for teaching help (explain a concept)

* Ask for a short explanation (2–3 sentences), then a medium (5–8 sentences), then a long explanation with math or diagrams.
* Request analogies specific to plants (e.g., "explain peak calling as if the ChIP experiment is a pollen trap measuring TF binding").
* Ask for a short quiz (3–5 questions) with answers and common mistakes students make.

---

## Example AI‑assisted grading rubric prompt

```
Prompt: "I have 12 student submissions producing a differential expression table. Each student submitted: (1) counts.csv, (2) code.R, (3) a 1‑page report. Create a grading rubric (0–5 per section) covering reproducibility, code clarity, correct design matrix, appropriate normalization, and interpretation. Provide model feedback comments for common errors (e.g., not accounting for batch effects, using raw counts in PCA)."
```

**Important:** Do not paste student identifiable info, Paths, or other information pertaining to the Hellbender cluster into public AI tools. If you need code review, anonymize submissions.

---

## Example full workflow cell (illustrative; **do not run without checking paths & versions**)

```bash
# Example: quick local dry-run for one sample (do not run blindly)
set -euo pipefail
SAMPLE=sample1
R1=./fastq/${SAMPLE}_R1.fastq.gz
R2=./fastq/${SAMPLE}_R2.fastq.gz
TRIM_OUT=./trimmed/${SAMPLE}
INDEX=$PROJECT_DIR/references/salmon_index
# dry-run print
echo "Would run: salmon quant -i ${INDEX} -l A -1 ${R1} -2 ${R2} -p 8 -o ${TRIM_OUT}"

# version checks
salmon --version || true
```

---

## Practical exercises (use these to practice forming AI prompts)

1. **Write a prompt** to get a Slurm script that runs `fastp`, `salmon`, and then aggregates results into a single `tximport` input table. Include `--dryrun` functionality. Submit your prompt and the AI output (as a plain text file) to the assignment box.
2. **Debugging task**: Run a purposely broken STAR command (provided in the next cell). Paste the exact error output and your AI prompt for debugging, then summarize what you learned.

---

## Appendix: Short prompt cheat‑sheet (copy & edit)

* "I am on an HPC slurm. Provide an sbatch script to run [tool] v[version] on paired FASTQ files in [path] using [N] cores and [MEM] memory. Output to [path]. Include version checks, a dry‑run mode, and a summary checklist of outputs. Also explain each line in plain English."

* "Explain in 3 sentences, 1 paragraph, and 1 page what [concept] is (e.g. pseudomapping, FDR, peak calling), then give an analogy suitable for plant scientists and 3 short questions I can quiz students with."

---

## Final notes for students

* Students must include the final AI prompt and the model/date in their submitted notebooks.
* Students are encouraged to annotate AI‑generated code with `# AI:` comments and to justify any edits.
---


# Prompt #2 (Bad Prompting): 
*"how do I prompt ai for coding?"*

# GPT-5 Response #2: 

# 🧠 How to Prompt AI for Coding

Prompting AI effectively can make a huge difference in the quality of help you get. Whether you're writing code, debugging, or learning a new concept, follow these guidelines:

---

## 🔧 1. Be Specific

**Instead of:**

> Help me with Python

**Try:**

> Write a Python function that takes a list of integers and returns only the prime numbers.

---

## 🧩 2. Include Context

If you're stuck, provide:

- A code snippet
- What it’s supposed to do
- What’s going wrong

**Example:**

```javascript
fetch('https://api.example.com/data')
  .then(response => response.json())
```
  
## 🧪 3. Ask for Examples or Explanations

Example prompts:

Explain how recursion works in Python with a factorial example.

Give me an example of a SQL query that joins two tables and filters by date.

🛠 4. Use Directives

Guide the AI with clear instructions

* "Write a function that..."

* "Refactor this code to use..."

* "Comment this code"

* "Explain what this code does"

Example:

Refactor this Python function to use list comprehension instead of a for loop.

## 📋 5. For Larger Tasks: Break It Down

Tackle multi-step problems gradually.

Example:

* Help me build a simple to-do list app using HTML, CSS, and JavaScript.
Start with the HTML structure.

Then:

* Now add basic styles with CSS.
* Now make the 'Add' button functional using JavaScript.

## 🧠 Bonus: Sample Prompts

✅ Write a Python script that scrapes all article titles from a given website using BeautifulSoup.

✅ Convert this Java code to Python: [insert code]

✅ Add error handling to this function.

✅ What are some security best practices for writing a login system in PHP?
    
*To demonstrate the power of prompting quickly and successfully, prompts within this notebook were generated by OpenAI's GPT-5 model using the prompt:*

# Final Notes: 
Please be sure to observe both prompts and see the applicability of the information provided. The first prompt - which was reviewed and contains accurate information on how to prompt AI for this course - is detailed and can be applied. The second prompt is full of filler and cannot be applied to this course. 

Overall, please be careful how you utilize AI for this course. It is okay to consult AI, especially if this is your first time programming, but it is necessary that you are able to parse code and understand what it is doing, even if you don't know the syntax or language. 